# Chronos-2 Time Series Forecasting Demo

基于 AutoGluon TimeSeries 原生 API，遵循官方教程：
https://auto.gluon.ai/stable/tutorials/timeseries/forecasting-chronos.html

支持：
- 零样本预测 (Zero-Shot)
- 微调迁移学习 (Fine-tuning with LoRA)
- 对比实验

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from ett_loader import prepare_ett_for_chronos
from chronos2_experiment import run_zero_shot, run_finetune, run_compare

print("Chronos-2 Demo Ready!")

## 1. 加载 ETT 数据集

In [ ]:
DATA_DIR = '/root/datasets/ett-dataset'
MODEL_PATH = '/root/chronos-2-model'
PREDICTION_LENGTH = 96

train, val, test = prepare_ett_for_chronos(
    dataset_name='ETTh1',
    prediction_length=PREDICTION_LENGTH,
    data_dir=DATA_DIR
)

print(f"Train: {train.shape}, Val: {val.shape}, Test: {test.shape}")
print(f"Items: {train.num_items}")
print(f"Columns: {list(train.columns)}")

## 2. 可视化数据

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
axes = axes.flatten()

for idx, item_id in enumerate(train.item_ids[:4]):
    ax = axes[idx]
    item_data = train.loc[item_id]
    ax.plot(item_data.index, item_data['target'], label='Train')
    
    test_data = test.loc[item_id]
    ax.plot(test_data.index, test_data['target'], label='Test', color='green')
    
    ax.set_title(f'Item: {item_id}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. 零样本预测 (Zero-Shot)

直接使用预训练 Chronos-2 模型，无需微调。

In [ ]:
zs_results = run_zero_shot(
    train_data=train,
    test_data=test,
    model_path=MODEL_PATH,
    prediction_length=PREDICTION_LENGTH,
    time_limit=120,
    results_dir='../results/notebook_zero_shot',
    plot=True,
)

## 4. 微调迁移学习 (Fine-tuning)

使用 LoRA 适配器高效微调 Chronos-2，默认不保存完整模型文件以节省空间。

In [ ]:
ft_results = run_finetune(
    train_data=train,
    val_data=val,
    test_data=test,
    model_path=MODEL_PATH,
    prediction_length=PREDICTION_LENGTH,
    time_limit=600,
    fine_tune_mode='lora',  # 'lora' 高效微调 或 'full' 全量微调
    fine_tune_lr=1e-3,
    fine_tune_steps=1000,
    results_dir='../results/notebook_finetune',
    plot=True,
)

## 5. 对比实验

同时运行零样本和微调模型，自动生成对比表格。

In [ ]:
compare_results = run_compare(
    train_data=train,
    val_data=val,
    test_data=test,
    model_path=MODEL_PATH,
    prediction_length=PREDICTION_LENGTH,
    time_limit=900,
    results_dir='../results/notebook_compare',
    plot=True,
)

## 6. AutoGluon 原生 API 直接调用

In [ ]:
from autogluon.timeseries import TimeSeriesPredictor

# 零样本
predictor = TimeSeriesPredictor(prediction_length=PREDICTION_LENGTH).fit(
    train,
    hyperparameters={
        'Chronos2': {'model_path': MODEL_PATH}
    },
    enable_ensemble=False,
)

predictions = predictor.predict(train)
print(predictions.head())

In [ ]:
# 微调
predictor_ft = TimeSeriesPredictor(prediction_length=PREDICTION_LENGTH).fit(
    train,
    hyperparameters={
        'Chronos2': {
            'model_path': MODEL_PATH,
            'fine_tune': True,
            'fine_tune_mode': 'lora',
        }
    },
    enable_ensemble=False,
)

predictions_ft = predictor_ft.predict(train)
print(predictions_ft.head())